## Quick Start

If you are using Google Colab:

1. install dependencies,
2. mount Google Drive if needed,
3. authenticate BigQuery,
4. set your project and dataset variables,
5. run all cells in order.

If you do not have MIMIC-IV and BigQuery access, you will not be able to reproduce this notebook end-to-end.

In [1]:
%pip install -q pandas numpy scikit-learn requests joblib tqdm

Note: you may need to restart the kernel to use updated packages.


## BigQuery Authentication and Dataset Setup

This section authenticates access to Google BigQuery and creates a working dataset for intermediate outputs.

- In Google Colab, authentication is done interactively using `auth.authenticate_user()`.
- In a local environment, Google Application Default Credentials must already be configured.
- This notebook does not store credentials directly.

In [2]:
%pip install -q python-dotenv google-cloud-bigquery

from dotenv import load_dotenv
import os

load_dotenv()

project = os.getenv("GCP_PROJECT_ID", "your-project-id")
DATASET_ID = os.getenv("BQ_DATASET_ID", "icd_project")

# Colab auth if available
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Google Colab.")
except ImportError:
    print("Not running in Colab. Expecting local ADC credentials.")

from google.cloud import bigquery

client = bigquery.Client(project=project)

dataset_ref = f"{project}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "US"

dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {dataset.full_dataset_id}")

Note: you may need to restart the kernel to use updated packages.
Not running in Colab. Expecting local ADC credentials.


c:\Users\Bhoomi\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Dataset ready: mimic-491022:icd_project


In [3]:
# Write a sample query to check connection
query = """
SELECT
  subject_id,
  hadm_id,
  charttime,
  text
FROM `physionet-data.mimiciv_note.discharge`
LIMIT 10
"""

In [4]:
# Run the query
query_job = client.query(query)

# Get the results
results = query_job.result()

In [5]:
# Process and extract the data
for row in results:
  print(row) # process the row data

Row((10011466, 21473984, datetime.datetime(2191, 8, 30, 0, 0), " \nName:  ___                  Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   M\n \nService: SURGERY\n \nAllergies: \nmorphine\n \nAttending: ___.\n \nChief Complaint:\nabdominal pain\n \nMajor Surgical or Invasive Procedure:\nnone\n\n \nHistory of Present Illness:\nThis patient is a ___ year old male who complains of RIGHT\nSIDED ABDOMINAL PAIN. Patient presents with 2 days of right\nlower quadrant pain. Patient states noticed it while\nwalking. Patient's noticed intermittent pain worsens.\nPatient had no relief with Pepto-Bismol. Patient denies\nfevers or chills. Patient reports some anorexia.\n \n\n \nPast Medical History:\nnone\n \nSocial History:\n___\nFamily History:\nNC\n \nPhysical Exam:\nPHYSICAL EXAMINATION:  upon admission: ___ \n \nTemp: 97.8 HR: 90 BP: 124/86 Resp: 14 O(2)Sat: 100\n \nConstitutional: Comfortable\nHEENT: Normocephalic, atrau

In [6]:
query = f"""
CREATE OR REPLACE TABLE `{project}.{DATASET_ID}.icd_base` AS
WITH discharge_notes AS (
  SELECT
    subject_id,
    hadm_id,
    charttime,
    text
  FROM `physionet-data.mimiciv_note.discharge`
  WHERE hadm_id IS NOT NULL
),
first_hadm AS (
  SELECT
    subject_id,
    MIN(admittime) AS first_admittime
  FROM `physionet-data.mimiciv_3_1_hosp.admissions`
  GROUP BY subject_id
),
first_hospitalizations AS (
  SELECT
    a.subject_id,
    a.hadm_id
  FROM `physionet-data.mimiciv_3_1_hosp.admissions` a
  JOIN first_hadm f
    ON a.subject_id = f.subject_id
   AND a.admittime = f.first_admittime
),
diag_codes AS (
  SELECT
    hadm_id,
    CONCAT('D_', CAST(icd_version AS STRING), '_', icd_code) AS full_code
  FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd`
  WHERE icd_code IS NOT NULL
),
proc_codes AS (
  SELECT
    hadm_id,
    CONCAT('P_', CAST(icd_version AS STRING), '_', icd_code) AS full_code
  FROM `physionet-data.mimiciv_3_1_hosp.procedures_icd`
  WHERE icd_code IS NOT NULL
),
all_codes AS (
  SELECT * FROM diag_codes
  UNION ALL
  SELECT * FROM proc_codes
)
SELECT
  n.subject_id,
  n.hadm_id,
  ANY_VALUE(n.charttime) AS charttime,
  ANY_VALUE(n.text) AS discharge_summary,
  ARRAY_AGG(DISTINCT c.full_code ORDER BY c.full_code) AS codes
FROM discharge_notes n
JOIN first_hospitalizations fh
  ON n.subject_id = fh.subject_id
 AND n.hadm_id = fh.hadm_id
JOIN all_codes c
  ON n.hadm_id = c.hadm_id
GROUP BY n.subject_id, n.hadm_id
"""
client.query(query).result()
print("Created icd_base")

Created icd_base


In [7]:
query = f"""
CREATE OR REPLACE TABLE `{project}.{DATASET_ID}.top50_codes` AS
SELECT
  code,
  COUNT(*) AS freq
FROM `{project}.{DATASET_ID}.icd_base`,
UNNEST(codes) AS code
GROUP BY code
ORDER BY freq DESC
LIMIT 50
"""
client.query(query).result()
print("Created top50_codes")

Created top50_codes


In [8]:
query = f"""
CREATE OR REPLACE TABLE `{project}.{DATASET_ID}.icd_top50` AS
WITH exploded AS (
  SELECT
    b.subject_id,
    b.hadm_id,
    b.charttime,
    b.discharge_summary,
    code
  FROM `{project}.{DATASET_ID}.icd_base` b,
  UNNEST(b.codes) AS code
),
filtered AS (
  SELECT
    e.subject_id,
    e.hadm_id,
    e.charttime,
    e.discharge_summary,
    e.code
  FROM exploded e
  JOIN `{project}.{DATASET_ID}.top50_codes` t
    ON e.code = t.code
)
SELECT
  subject_id,
  hadm_id,
  ANY_VALUE(charttime) AS charttime,
  ANY_VALUE(discharge_summary) AS discharge_summary,
  ARRAY_AGG(DISTINCT code ORDER BY code) AS top50_codes
FROM filtered
GROUP BY subject_id, hadm_id
"""
client.query(query).result()
print("Created icd_top50")

Created icd_top50


In [9]:
query = f"""
CREATE OR REPLACE TABLE `{project}.{DATASET_ID}.icd_top50_clean` AS
SELECT *
FROM `{project}.{DATASET_ID}.icd_top50`
WHERE ARRAY_LENGTH(top50_codes) > 0
  AND discharge_summary IS NOT NULL
  AND LENGTH(TRIM(discharge_summary)) > 0
"""
client.query(query).result()
print("Created icd_top50_clean")

Created icd_top50_clean


In [10]:
%pip install db-dtypes
%pip install db_dtypes
import db_dtypes
query = f"""
SELECT
  COUNT(*) AS n_rows,
  AVG(ARRAY_LENGTH(top50_codes)) AS avg_labels_per_note,
  AVG(LENGTH(discharge_summary)) AS avg_chars,
  MAX(LENGTH(discharge_summary)) AS max_chars
FROM `{project}.{DATASET_ID}.icd_top50_clean`
"""
stats_df = client.query(query).to_dataframe()
stats_df

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


,n_rows,avg_labels_per_note,avg_chars,max_chars
0,110626,3.38895,10231.37079,58596


In [11]:
query = f"""
SELECT
  code,
  COUNT(*) AS freq
FROM `{project}.{DATASET_ID}.icd_top50_clean`,
UNNEST(top50_codes) AS code
GROUP BY code
ORDER BY freq DESC
"""
freq_df = client.query(query).to_dataframe()
freq_df.head(20)

,code,freq
0,D_9_4019,34750
1,D_9_2724,22047
2,D_10_I10,16139
3,D_9_53081,14144
4,D_10_E785,13930
5,D_9_25000,12562
6,D_9_41401,11973
7,D_9_42731,11528
8,D_10_Z87891,10654
9,D_9_V1582,9535


In [12]:
query = f"""
SELECT *
FROM `{project}.{DATASET_ID}.icd_top50_clean`
LIMIT 5
"""
sample_df = client.query(query).to_dataframe()
sample_df.head()

,subject_id,hadm_id,charttime,discharge_summary,top50_codes
0,16388955,28985645,2134-05-11,\nName: ___ Unit No: ___\...,"[D_10_E669, D_10_E785, D_10_I10, D_10_I4891, D..."
1,15692407,23125765,2149-02-03,\nName: ___ Unit No: ___...,[D_9_2724]
2,14915965,23153477,2148-03-18,\nName: ___ Unit No: __...,"[D_9_2724, D_9_2859, D_9_4019, D_9_4280, D_9_4..."
3,19864570,25619354,2169-12-09,\nName: ___ Unit No: ___\n ...,"[D_9_30000, D_9_311, D_9_49390]"
4,17610001,26613751,2185-12-04,\nName: ___ Unit No: ___\...,[D_9_3051]


In [13]:
query = f"""
SELECT
  subject_id,
  hadm_id,
  discharge_summary,
  top50_codes
FROM `{project}.{DATASET_ID}.icd_top50_clean`
"""
df = client.query(query).to_dataframe()
print(df.shape)
df.head()

(110626, 4)


,subject_id,hadm_id,discharge_summary,top50_codes
0,11331748,28794813,\nName: ___. Unit No: ___\...,"[D_9_496, D_9_53081]"
1,10086841,21969733,\nName: ___ Unit No: ___...,"[D_9_32723, D_9_4019, D_9_49390, D_9_53081]"
2,13401529,24039724,\nName: ___ Unit No: ___\...,"[D_9_2449, D_9_4019]"
3,12660752,27398847,\nName: ___ Unit No: _...,"[D_9_2724, D_9_2761, D_9_40390]"
4,16978870,26299619,\nName: ___ Unit No: ___\n \n...,"[D_9_311, D_9_42789, D_9_486]"


In [14]:
df.to_pickle("icd_top50_dataset.pkl")

In [15]:
import re

def clean_and_truncate(text, max_words=500):  # reduce from 2500 → 500
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    return " ".join(words[:max_words])

df["text"] = df["discharge_summary"].apply(clean_and_truncate)

# Drop original column to save memory
df = df.drop(columns=["discharge_summary"])

In [16]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["top50_codes"])

print("Number of labels:", len(mlb.classes_))
print("First 10 labels:", mlb.classes_[:10])

Number of labels: 50
First 10 labels: ['D_10_D62' 'D_10_D649' 'D_10_E039' 'D_10_E119' 'D_10_E669' 'D_10_E785'
 'D_10_F17210' 'D_10_F329' 'D_10_F419' 'D_10_I10']


In [17]:
from sklearn.model_selection import GroupShuffleSplit

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df["subject_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["subject_id"]))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print(train_df.shape, val_df.shape, test_df.shape)

(88500, 4) (11063, 4) (11063, 4)


In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform(train_df["top50_codes"])
Y_val = mlb.transform(val_df["top50_codes"])
Y_test = mlb.transform(test_df["top50_codes"])

label_names = list(mlb.classes_)
len(label_names), label_names[:10]

(50,
 ['D_10_D62',
  'D_10_D649',
  'D_10_E039',
  'D_10_E119',
  'D_10_E669',
  'D_10_E785',
  'D_10_F17210',
  'D_10_F329',
  'D_10_F419',
  'D_10_I10'])

In [19]:
import joblib

joblib.dump(mlb, "label_binarizer.pkl")

train_df.to_pickle("train_df.pkl")
val_df.to_pickle("val_df.pkl")
test_df.to_pickle("test_df.pkl")

joblib.dump(Y_train, "Y_train.pkl")
joblib.dump(Y_val, "Y_val.pkl")
joblib.dump(Y_test, "Y_test.pkl")

['Y_test.pkl']